# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

**Objective:** membandingkan dua pendekatan untuk binary sentiment analysis terhadap review customer:

1. Model klasik Machine Learning yang dilatih sendiri menggunakan Scikit-learn.
2. Large Language Model (Gemini API) yang digunakan dengan pendekatan prompting.

Kedua pendekatan dievaluasi pada **test set yang sama** menggunakan Accuracy, Precision, Recall, dan F1-Score, lalu dibandingkan dari sisi performa, waktu implementasi/inference, dan biaya API.

**Target Label:**

- `positif` → review yang menunjukkan pengalaman, kualitas, atau kepuasan positif.
- `negatif` → review yang menunjukkan kritik, masalah, kekurangan, atau pengalaman negatif.

**Asumsi utama:**

- Tidak ada kelas netral.
- Satu review memiliki satu label sentiment.
- `review_text` adalah sumber informasi utama untuk klasifikasi.
- Untuk baseline ML digunakan TF-IDF + Logistic Regression.
- Gemini digunakan zero-shot dengan instruksi klasifikasi yang eksplisit dan output dibatasi ke `positif` atau `negatif`.

## 2. Import Library & Load Dataset

In [19]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
from tqdm.auto import tqdm
from collections import deque
import random
import time
import re
import os
from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY = os.getenv("API_KEYS")

MAX_RPM = 10
WINDOW = 60

In [8]:
df = pd.read_csv('../data/customer_reviews_sentiment.csv')
df.head()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [9]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

Train size: 160
Test size: 40


## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [10]:
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1,)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [11]:
train_start = time.perf_counter()

clf = LogisticRegression(random_state=67, max_iter=1000)
clf.fit(X_train_vec, y_train)

train_time = time.perf_counter() - train_start
print(f"Training time: {train_time:.6f} seconds")

Training time: 0.008672 seconds


### 4.3 Prediksi pada Test Set

In [12]:
classic_inference_start = time.perf_counter()
y_pred_classic = clf.predict(X_test_vec)
classic_inference_time = time.perf_counter() - classic_inference_start

print(f"Classic inference time ({len(X_test)} reviews): {classic_inference_time:.6f} seconds")
print(f"Classic average latency/review: {classic_inference_time / len(X_test):.6f} seconds")


Classic inference time (40 reviews): 0.000572 seconds
Classic average latency/review: 0.000014 seconds


## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API

In [13]:
# Simpan API key di environment variable, JANGAN hardcode langsung di notebook
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

### 5.2 Merancang Prompt

_Tuliskan prompt yang kamu rancang di sini, dan jelaskan alasannya (zero-shot / few-shot)._

In [18]:
request_times = deque()

def wait_for_rate_limit():
    now = time.monotonic()

    while request_times and now - request_times[0] >= WINDOW:
        request_times.popleft()

    if len(request_times) >= MAX_RPM:
        wait_time = WINDOW - (now - request_times[0])
        time.sleep(wait_time)

    request_times.append(time.monotonic())

In [22]:
# 🔧 Sesuaikan label pada prompt jika dataset final menggunakan label berbeda
def classify_sentiment_llm(review_text, max_retries=3):
    prompt = f'''Kamu adalah data annotator yang handal dalam mengkategorikan ulasan customer

Tugas: 

Klasifikasikan ulasan customer ke dalam SATU kategori :
- positif = pengalaman/kualitas/kepuasan customer cenderung positif
- negatif = kritik/masalah/kekurangan/pengalaman customer cenderung negatif

Format:
1. Jawab hanya dengan satu kata: positif atau negatif.
2. Jangan berikan alasan, tanda baca, atau teks tambahan.

Ulasan: "{review_text}"
Sentimen:'''

    for attempt in range(max_retries):
        try:
            wait_for_rate_limit()
            start = time.perf_counter()

            response = client.models.generate_content(
                model='gemini-3.1-flash-lite',  # sesuaikan versi model
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0
                )
            )

            api_latency = time.perf_counter() - start
            raw_text = getattr(response, "text", "") or ""
            label = normalize_llm_label(raw_text)

            return label, raw_text, response, api_latency

        except Exception as e:
            error_text = str(e)

            if "429" not in error_text:
                raise

            wait_time = min(60, 3 ** attempt + random.uniform(1, 3))

            print(
                f"Rate limit reached. "
                f"Retry {attempt + 1}/{max_retries} "
                f"after {wait_time:.1f}s..."
            )

            time.sleep(wait_time)
            
    raise RuntimeError("Gemini API failed after maximum retries")

### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [15]:
def normalize_llm_label(text):
    """Normalisasi output Gemini menjadi 'positif'/'negatif' atau None."""
    if text is None:
        return None

    cleaned = str(text).strip().lower()
    cleaned = re.sub(r"[^a-zA-ZÀ-ÿ\s]", " ", cleaned)
    cleaned = " ".join(cleaned.split())

    if cleaned in {"positif", "negatif"}:
        return cleaned

    if "positif" in cleaned and "negatif" not in cleaned:
        return "positif"
    if "negatif" in cleaned and "positif" not in cleaned:
        return "negatif"

    return None

In [23]:
y_pred_llm = []
llm_raw_outputs = []
llm_latencies = []
llm_usage = []

for review in tqdm(X_test, desc="Gemini API classification"):

    label, raw_text, response, api_latency = classify_sentiment_llm(review)

    y_pred_llm.append(label)
    llm_raw_outputs.append(raw_text)
    llm_latencies.append(api_latency)

    usage = getattr(response, "usage_metadata", None)
    llm_usage.append(usage)

    time.sleep(8)

llm_results_preview = pd.DataFrame({
    "review_text": X_test.reset_index(drop=True),
    "actual": y_test.reset_index(drop=True),
    "raw_output": llm_raw_outputs,
    "prediction": y_pred_llm,
    "latency_sec": llm_latencies,
})

display(llm_results_preview.head(10))

invalid_count = sum(pred not in {"positif", "negatif"} for pred in y_pred_llm)
print("Invalid Gemini outputs:", invalid_count)

if invalid_count:
    print("Review baris invalid sebelum evaluasi.")

Gemini API classification: 100%|██████████| 40/40 [06:37<00:00,  9.94s/it]


,review_text,actual,raw_output,prediction,latency_sec
0,Barang tidak berfungsi sama sekali begitu dite...,negatif,negatif,negatif,1.228900
1,"Sudah langganan di toko ini, kualitas selalu k...",positif,positif,positif,1.240266
2,"Suka banget sama warnanya, sesuai ekspektasi.",positif,positif,positif,1.116132
3,"Kualitas produk sangat memuaskan, akan order l...",positif,positif,positif,4.009924
4,"Ukuran tidak sesuai, terlalu kecil dari yang d...",negatif,negatif,negatif,1.645525
5,Sudah bayar mahal tapi kualitas jauh dari eksp...,negatif,negatif,negatif,1.049726
6,"Suka banget sama warnanya, sesuai ekspektasi.",positif,positif,positif,2.588076
7,"Fast respon, pengiriman cepat, barang aman sam...",positif,positif,positif,1.211179
8,"Produk palsu, bukan barang original seperti ya...",negatif,negatif,negatif,1.332821
9,Komplain tidak ditanggapi sama sekali oleh pen...,negatif,negatif,negatif,1.429146


Invalid Gemini outputs: 0


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [25]:
print("=== Model Klasik (Scikit-learn) ===")
classic_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred_classic),
    "Precision": precision_score(y_test, y_pred_classic, pos_label="positif", zero_division=0),
    "Recall": recall_score(y_test, y_pred_classic, pos_label="positif", zero_division=0),
    "F1-Score": f1_score(y_test, y_pred_classic, pos_label="positif", zero_division=0),
}

display(pd.DataFrame([classic_metrics], index=["Scikit-learn"]))

print("\nClassification report:")
print(classification_report(y_test, y_pred_classic, labels=["negatif", "positif"], digits=4))

print("Confusion Matrix [negatif, positif]:")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred_classic, labels=["negatif", "positif"]),
    index=["actual_negatif", "actual_positif"],
    columns=["pred_negatif", "pred_positif"],
))


=== Model Klasik (Scikit-learn) ===


,Accuracy,Precision,Recall,F1-Score
Scikit-learn,1.0,1.0,1.0,1.0



Classification report:
              precision    recall  f1-score   support

     negatif     1.0000    1.0000    1.0000        18
     positif     1.0000    1.0000    1.0000        22

    accuracy                         1.0000        40
   macro avg     1.0000    1.0000    1.0000        40
weighted avg     1.0000    1.0000    1.0000        40

Confusion Matrix [negatif, positif]:


,pred_negatif,pred_positif
actual_negatif,18,0
actual_positif,0,22


### 6.2 Evaluasi LLM API

In [26]:
if len(y_pred_llm) != len(y_test):
    raise ValueError("Jumlah prediksi Gemini tidak sama dengan jumlah test set.")

valid_mask = pd.Series(y_pred_llm, index=y_test.index).isin(["positif", "negatif"])
if not valid_mask.all():
    print("Evaluasi Gemini dihentikan karena terdapat output yang tidak dapat dipetakan ke label.")
    print("Jumlah invalid:", int((~valid_mask).sum()))
else:
    print("=== LLM API (Gemini) ===")

    llm_metrics = {
        "Accuracy": accuracy_score(y_test, y_pred_llm),
        "Precision": precision_score(y_test, y_pred_llm, pos_label="positif", zero_division=0),
        "Recall": recall_score(y_test, y_pred_llm, pos_label="positif", zero_division=0),
        "F1-Score": f1_score(y_test, y_pred_llm, pos_label="positif", zero_division=0),
    }

    display(pd.DataFrame([llm_metrics], index=["Gemini"]))

    print("\nClassification report:")
    print(classification_report(y_test, y_pred_llm, labels=["negatif", "positif"], digits=4))

    print("Confusion Matrix [negatif, positif]:")
    display(pd.DataFrame(
        confusion_matrix(y_test, y_pred_llm, labels=["negatif", "positif"]),
        index=["actual_negatif", "actual_positif"],
        columns=["pred_negatif", "pred_positif"],
    ))

    print(f"\nTotal Gemini inference time: {sum(llm_latencies):.4f} seconds")
    print(f"Average Gemini latency/review: {sum(llm_latencies)/len(llm_latencies):.4f} seconds")


=== LLM API (Gemini) ===


,Accuracy,Precision,Recall,F1-Score
Gemini,1.0,1.0,1.0,1.0



Classification report:
              precision    recall  f1-score   support

     negatif     1.0000    1.0000    1.0000        18
     positif     1.0000    1.0000    1.0000        22

    accuracy                         1.0000        40
   macro avg     1.0000    1.0000    1.0000        40
weighted avg     1.0000    1.0000    1.0000        40

Confusion Matrix [negatif, positif]:


,pred_negatif,pred_positif
actual_negatif,18,0
actual_positif,0,22



Total Gemini inference time: 77.6472 seconds
Average Gemini latency/review: 1.9412 seconds


### 6.3 Tabel Perbandingan Ringkasan

_Susun tabel ringkasan (bisa markdown table atau DataFrame) yang membandingkan Accuracy, Precision, Recall, F1-Score kedua pendekatan._

In [29]:
comparison_df = pd.DataFrame([
    {"Approach": "Scikit-learn", **classic_metrics},
    {"Approach": "Gemini", **llm_metrics},
]).set_index("Approach")

display(comparison_df)

print("Latency:")
print(f"- Scikit-learn total: {classic_inference_time:.6f} sec")
print(f"- Scikit-learn avg/review: {classic_inference_time/len(X_test):.6f} sec")
print(f"- Gemini total: {sum(llm_latencies):.6f} sec")
print(f"- Gemini avg/review: {sum(llm_latencies)/len(llm_latencies):.6f} sec")


,Accuracy,Precision,Recall,F1-Score
Approach,,,,
Scikit-learn,1.0,1.0,1.0,1.0
Gemini,1.0,1.0,1.0,1.0


Latency:
- Scikit-learn total: 0.000572 sec
- Scikit-learn avg/review: 0.000014 sec
- Gemini total: 77.647170 sec
- Gemini avg/review: 1.941179 sec


## 7. Analisis Trade-off dan Limitation

_Tuliskan analisis: pendekatan mana yang lebih unggul dari sisi performa, trade-off effort/kecepatan/biaya, dan keterbatasan masing-masing pendekatan._

### 7.1 Performa
Kedua pendekatan memiliki skor akurasi, precision, recall, dan f1 score sama. Namun, pendekatan model ML secara latensi jauh lebih rendah

**Catatan penting:** dataset hanya berisi 200 review dan test set hanya 40 review. Karena ukuran test set kecil, satu prediksi yang berubah dapat menggeser metrik sekitar 2,5 percentage points. Hasil ini sebaiknya dianggap sebagai hasil eksperimen pada dataset ini, bukan generalisasi untuk seluruh populasi customer.

### 7.2 Implementasi dan maintenance
- **Scikit-learn:** membutuhkan preprocessing, training, evaluasi, dan maintenance model. Setelah model terlatih, inference lokal relatif sederhana.
- **Gemini:** implementasi klasifikasi dapat lebih cepat karena tidak perlu melatih model, tetapi aplikasi bergantung pada API, credential, network, quota/rate limit, dan perubahan model/API.

### 7.3 Biaya
Model klasik tidak memiliki biaya API per prediction setelah infrastruktur tersedia, sedangkan Gemini memiliki biaya berbasis token pada paid tier.

### 7.4 Limitations
1. Dataset kecil (200 baris) sehingga hasil belum cukup untuk menyimpulkan performa production-scale.
2. Split hanya satu kali (`random_state=42`); belum ada repeated cross-validation atau confidence interval.
3. Baseline ML hanya menggunakan satu algoritma (Logistic Regression) dan satu skema TF-IDF.
4. Gemini dievaluasi dengan satu prompt dan satu model, sehingga hasil dapat berubah jika prompt/model/parameter diubah.
5. LLM API memiliki faktor eksternal seperti latency, quota, network failure, dan perubahan model.


## 8. Rekomendasi Technical Approach

_Tuliskan rekomendasi akhir beserta alasannya, berdasarkan hasil eksperimen di atas._

 Pada dataset ini, kedua pendekatan menghasilkan Accuracy, Precision, Recall, dan F1-Score sempurna, walaupun terdapat limitasi dataset seperti yang telah disebutkan. Dengan mempertimbangkan kebutuhan use case yaitu menampilkan klasifikasi sentimen ulasan pada halaman produk, technical approach yang dipilih adalah menggunakan model ML Scikit-Learn karena secara performa setara dengan LLM Gemini API tetapi tidak membutuhkan biaya dan latensi yang jauh lebih rendah, dengan catatan bahwa hasil eksperimen terbatas pada dataset dan konfigurasi yang digunakan.